# Tutorial 10: Audio Generation with Telecommunications Backend

<p align="right" style="margin-right: 8px;">
    <a target="_blank" href="https://colab.research.google.com/github/idiap/sdialog/blob/main/tutorials/01_audio/2.accoustic_simulation.ipynb">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
    </a>
</p>

## Getting started

This tutorial demonstrates how to simulate a telephone call where two speakers are located in completely different rooms. We will use the new `telecommunications` backend which applies room acoustics individually and then simulates a telephone channel.

### Environment Setup

Let's first check if our environment is all set up.

In [ ]:
# Setup the environment depending on weather we are running in Google Colab or Jupyter Notebook
import os
from IPython import get_ipython
from IPython.display import Audio, display

if "google.colab" in str(get_ipython()):
    print("Running on CoLab")

    !sudo apt-get install sox ffmpeg
    !sudo apt-get -qq -y install espeak-ng > /dev/null 2>&1
    %pip install -q kokoro>=0.9.4
    # Installing sdialog
    !git clone https://github.com/idiap/sdialog.git
    %cd sdialog
    %pip install -e .[audio]
    %cd ..
else:
    print("Running in Jupyter Notebook")

### Load the example dialogue

In order to run the next steps in a fast manner, we will start from an existing dialog generated using previous tutorials:

In [ ]:
from sdialog import Dialog

path_dialog = "../../tests/data/demo_dialog_doctor_patient.json"

if not os.path.exists(path_dialog):
    !wget https://raw.githubusercontent.com/idiap/sdialog/refs/heads/main/tests/data/demo_dialog_doctor_patient.json
    path_dialog = "./demo_dialog_doctor_patient.json"

original_dialog = Dialog.from_file(path_dialog)
original_dialog.print()

# Tutorial

### Instanciate voices database

In [ ]:
from sdialog.audio.voice_database import HuggingfaceVoiceDatabase
kokoro_voice_database = HuggingfaceVoiceDatabase("sdialog/voices-kokoro")

### Instanciate TTS model

In [ ]:
from sdialog.audio.tts import KokoroTTS
tts_engine = KokoroTTS()

## Setup stage: Audio Dialog and Audio Pipeline

In [ ]:
from sdialog.audio.dialog import AudioDialog
from sdialog.audio.pipeline import AudioPipeline
from sdialog.audio.room import MicrophonePosition

Convert the original dialog into a audio enhanced dialog

In [ ]:
dialog: AudioDialog = AudioDialog.from_dialog(original_dialog)

In [ ]:
os.makedirs("./audio_outputs", exist_ok=True)
audio_pipeline = AudioPipeline(
    voice_database=kokoro_voice_database,
    tts_engine=tts_engine,
    dir_audio="./audio_outputs",
)

or if you encounter any issue during the download due to timeout:

In [ ]:
%%script false --no-raise-error
!hf download sdialog/background --repo-type dataset
!hf download sdialog/foreground --repo-type dataset

Now let's generate a medical room it will be enough and display it's shape and content:

In [ ]:
from sdialog.audio.room import DirectivityType
from sdialog.audio.utils import SourceVolume, SourceType
from sdialog.audio.room import SpeakerSide, Role, RoomPosition
from sdialog.audio.jsalt import MedicalRoomGenerator, RoomRole

In [ ]:
room_doctor = MedicalRoomGenerator().generate(args={"room_type": RoomRole.EXAMINATION})

In [ ]:
room_doctor.place_speaker_around_furniture(
    speaker_name=Role.SPEAKER_1, 
    furniture_name="desk", 
    max_distance=1.0, 
    side=SpeakerSide.FRONT
)
room_doctor.place_speaker_around_furniture(
    speaker_name=Role.SPEAKER_2, 
    furniture_name="desk", 
    max_distance=1.0, 
    side=SpeakerSide.FRONT
)
room_doctor.set_directivity(direction=DirectivityType.OMNIDIRECTIONAL)
room_doctor.set_mic_position(MicrophonePosition.CHEST_POCKET_SPEAKER_2)

In [ ]:
img_doctor = room_doctor.to_image()
display(img_doctor)
img_doctor.save("room_doctor.png")

In [ ]:
room_patient = MedicalRoomGenerator().generate(args={"room_type": RoomRole.EXAMINATION})

In [ ]:
room_patient.place_speaker_around_furniture(
    speaker_name=Role.SPEAKER_2, 
    furniture_name="desk", 
    max_distance=1.0,
    side=SpeakerSide.BACK
)
room_patient.place_speaker_around_furniture(
    speaker_name=Role.SPEAKER_1, 
    furniture_name="desk", 
    max_distance=1.0,
    side=SpeakerSide.BACK
)
room_patient.set_directivity(direction=DirectivityType.OMNIDIRECTIONAL)
room_patient.set_mic_position(MicrophonePosition.CHEST_POCKET_SPEAKER_2)

In [ ]:
img_patient = room_patient.to_image()
display(img_patient)
img_patient.save("room_patient.png")

We pass `room_acoustics_backend="telecom"` to the inference method.

We also provide the `speaker_rooms` mapping in the `environment` dictionary so the backend knows which room belongs to which speaker.

In [ ]:
dialog: AudioDialog = audio_pipeline.inference(
    dialog,
    perform_room_acoustics=True,
    room_acoustics_backend="telecom",
    add_sound_effects=False,
    environment={
        "speaker_rooms": {
            Role.SPEAKER_1: room_doctor,
            Role.SPEAKER_2: room_patient
        },
        # Speaker 1's environment
        "background_effect_speaker_1": "white_noise",
        "foreground_effect_speaker_1": "ac_noise_minimal",
        "foreground_effect_position_speaker_1": RoomPosition.TOP_RIGHT,
        
        # Speaker 2's environment
        "background_effect_speaker_2": "white_noise",
        # "foreground_effect_speaker_2": "ac_noise_minimal",
        "foreground_effect_position_speaker_2": RoomPosition.BOTTOM_LEFT,
        
        "source_volumes": {
            SourceType.ROOM: SourceVolume.HIGH,
            SourceType.BACKGROUND: SourceVolume.VERY_LOW
        },
        "kwargs_pyroom": {
            "ray_tracing": True,
            "air_absorption": True
        },
        "mixing_strategy": "stereo"
    },
    dialog_dir_name="demo_dialog_telecom",
    room_name="telephone_call_simulation",
    verbose=True
)

In [ ]:
final_audio_path = dialog.audio_step_3_filepaths["telephone_call_simulation"].audio_path
print(f"Audio saved at: {final_audio_path}")

In [ ]:
display(Audio(final_audio_path))